# RFM顧客セグメンテーション分析

ECサイトの注文データを用いて、RFM（Recency・Frequency・Monetary）で顧客をスコアリング・セグメント分類します。

---

**実行環境:** MySQL 8.0 / Python 3 / pandas  
**DB:** `sql_portfolio`（`SETUP.md` の手順で事前に構築）

## セットアップ

In [1]:
import mysql.connector
import pandas as pd
from IPython.display import display, HTML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

con = mysql.connector.connect(
    host='localhost', user='root', password='',
    database='sql_portfolio',
    charset='utf8mb4'
)

def run(sql):
    return pd.read_sql(sql, con)


---

## 分析クエリ

### 01. RFMスコア算出

**ビジネス課題:** 顧客ごとのR（最終購買日）・F（購買回数）・M（購買金額）を集計する。

**使用テクニック:** CTE, `DATEDIFF`, `CURDATE()`

In [2]:
sql = '''
WITH customer_orders AS (
    SELECT
        o.customer_id,
        MAX(o.order_date)                AS last_order_date,
        COUNT(DISTINCT o.order_id)       AS frequency,
        SUM(oi.quantity * oi.unit_price)  AS monetary
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid', 'shipped')
    GROUP BY o.customer_id
)
SELECT
    c.customer_id,
    c.customer_name,
    co.last_order_date,
    DATEDIFF(CURDATE(), co.last_order_date) AS recency_days,
    co.frequency,
    co.monetary
FROM customers c
JOIN customer_orders co ON c.customer_id = co.customer_id
ORDER BY co.monetary DESC;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_26396\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,customer_id,customer_name,last_order_date,recency_days,frequency,monetary
0,29,小林誠,2024-11-23 22:09:00,495,6,"59,900.00"
1,1,田中太郎,2024-12-22 11:25:00,466,3,"57,500.00"
2,5,高橋翔太,2024-11-02 12:23:00,516,5,"50,100.00"
3,13,松本健二,2024-05-22 18:23:00,680,4,"43,400.00"
4,4,山田大輔,2024-06-08 11:04:00,663,3,"42,500.00"
5,17,清水拓也,2024-10-09 10:59:00,540,3,"23,800.00"
6,27,伊藤遥,2024-11-12 17:35:00,506,4,"22,100.00"
7,25,高橋翔太,2024-04-17 22:00:00,715,2,"22,000.00"
8,8,中村和也,2024-06-09 18:31:00,662,3,"21,960.00"
9,20,前田浩,2024-10-19 09:04:00,530,4,"21,560.00"


### 02. RFMスコアの5段階スコアリング

**ビジネス課題:** R/F/M を5段階にスコアリングし、定量比較可能な指標へ変換する。

**使用テクニック:** `NTILE(5)` の多軸適用

In [3]:
sql = '''
WITH customer_rfm AS (
    SELECT
        o.customer_id,
        DATEDIFF(CURDATE(), MAX(o.order_date))  AS recency,
        COUNT(DISTINCT o.order_id)              AS frequency,
        SUM(oi.quantity * oi.unit_price)         AS monetary
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid', 'shipped')
    GROUP BY o.customer_id
),
scored AS (
    SELECT
        customer_id,
        recency,
        frequency,
        monetary,
        -- Recency は小さいほうが良いので DESC でスコア付け
        NTILE(5) OVER (ORDER BY recency DESC)   AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC)  AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC)   AS m_score
    FROM customer_rfm
)
SELECT
    s.customer_id,
    c.customer_name,
    s.recency,
    s.frequency,
    s.monetary,
    s.r_score,
    s.f_score,
    s.m_score,
    s.r_score + s.f_score + s.m_score AS rfm_total
FROM scored s
JOIN customers c ON s.customer_id = c.customer_id
ORDER BY rfm_total DESC, s.monetary DESC;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_26396\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,customer_id,customer_name,recency,frequency,monetary,r_score,f_score,m_score,rfm_total
0,29,小林誠,495,6,"59,900.00",4,5,5,14
1,1,田中太郎,466,3,"57,500.00",5,4,5,14
2,5,高橋翔太,516,5,"50,100.00",3,5,5,13
3,27,伊藤遥,506,4,"22,100.00",4,5,4,13
4,20,前田浩,530,4,"21,560.00",3,5,4,12
5,6,渡辺由美,462,3,"20,500.00",5,4,3,12
6,13,松本健二,680,4,"43,400.00",1,5,5,11
7,17,清水拓也,540,3,"23,800.00",3,4,4,11
8,21,田中彩,509,3,"14,000.00",3,4,3,10
9,7,伊藤拓也,496,3,"10,880.00",4,4,2,10


### 03. RFMセグメント分類

**ビジネス課題:** スコアの組み合わせから「VIP」「休眠リスク」「離脱」等に自動分類する。

**使用テクニック:** CTE 3段, `CASE` 多分岐

In [4]:
sql = '''
WITH customer_rfm AS (
    SELECT
        o.customer_id,
        DATEDIFF(CURDATE(), MAX(o.order_date))  AS recency,
        COUNT(DISTINCT o.order_id)              AS frequency,
        SUM(oi.quantity * oi.unit_price)         AS monetary
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid', 'shipped')
    GROUP BY o.customer_id
),
scored AS (
    SELECT
        customer_id,
        recency,
        frequency,
        monetary,
        NTILE(5) OVER (ORDER BY recency DESC)   AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC)  AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC)   AS m_score
    FROM customer_rfm
),
segmented AS (
    SELECT
        *,
        r_score + f_score + m_score AS rfm_total,
        CASE
            -- VIP: 最近も買い、頻度も金額も高い
            WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4
                THEN 'VIP'
            -- 優良顧客: スコア合計が高い
            WHEN r_score + f_score + m_score >= 11
                THEN '優良顧客'
            -- 新規有望: 最近買ったが回数はまだ少ない
            WHEN r_score >= 4 AND f_score <= 2
                THEN '新規有望'
            -- 休眠リスク: 以前は買っていたが最近来ていない
            WHEN r_score <= 2 AND f_score >= 3
                THEN '休眠リスク'
            -- 離脱: 長期間購買なし & 低頻度
            WHEN r_score <= 2 AND f_score <= 2
                THEN '離脱'
            ELSE 'その他'
        END AS segment
    FROM scored
)
SELECT
    seg.customer_id,
    c.customer_name,
    seg.r_score,
    seg.f_score,
    seg.m_score,
    seg.rfm_total,
    seg.segment
FROM segmented seg
JOIN customers c ON seg.customer_id = c.customer_id
ORDER BY seg.rfm_total DESC, seg.monetary DESC;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_26396\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,customer_id,customer_name,r_score,f_score,m_score,rfm_total,segment
0,29,小林誠,4,5,5,14,VIP
1,1,田中太郎,5,4,5,14,VIP
2,5,高橋翔太,3,5,5,13,優良顧客
3,27,伊藤遥,4,5,4,13,VIP
4,20,前田浩,3,5,4,12,優良顧客
5,6,渡辺由美,5,4,3,12,優良顧客
6,13,松本健二,1,5,5,11,優良顧客
7,17,清水拓也,3,4,4,11,優良顧客
8,21,田中彩,3,4,3,10,その他
9,7,伊藤拓也,4,4,2,10,その他


### 04. セグメント別集計レポート

**ビジネス課題:** セグメントごとの人数・平均LTV・平均購買間隔を集計する。

**使用テクニック:** `GROUP BY`, `AVG`, `NULLIF`

In [5]:
sql = '''
WITH customer_rfm AS (
    SELECT
        o.customer_id,
        DATEDIFF(CURDATE(), MAX(o.order_date))  AS recency,
        COUNT(DISTINCT o.order_id)              AS frequency,
        SUM(oi.quantity * oi.unit_price)         AS monetary
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid', 'shipped')
    GROUP BY o.customer_id
),
scored AS (
    SELECT
        customer_id, recency, frequency, monetary,
        NTILE(5) OVER (ORDER BY recency DESC)   AS r_score,
        NTILE(5) OVER (ORDER BY frequency ASC)  AS f_score,
        NTILE(5) OVER (ORDER BY monetary ASC)   AS m_score
    FROM customer_rfm
),
segmented AS (
    SELECT
        *,
        CASE
            WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'VIP'
            WHEN r_score + f_score + m_score >= 11             THEN '優良顧客'
            WHEN r_score >= 4 AND f_score <= 2                  THEN '新規有望'
            WHEN r_score <= 2 AND f_score >= 3                  THEN '休眠リスク'
            WHEN r_score <= 2 AND f_score <= 2                  THEN '離脱'
            ELSE 'その他'
        END AS segment
    FROM scored
)
SELECT
    segment,
    COUNT(*)                              AS customer_count,
    ROUND(AVG(monetary), 0)               AS avg_ltv,
    ROUND(AVG(frequency), 1)              AS avg_frequency,
    ROUND(AVG(recency), 0)                AS avg_recency_days,
    ROUND(AVG(monetary / NULLIF(frequency, 0)), 0) AS avg_order_value
FROM segmented
GROUP BY segment
ORDER BY avg_ltv DESC;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_26396\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,segment,customer_count,avg_ltv,avg_frequency,avg_recency_days,avg_order_value
0,VIP,3,"46,500.00",4.30,489.00,"11,558.00"
1,優良顧客,5,"31,872.00",3.80,546.00,"8,205.00"
2,休眠リスク,4,"24,990.00",3.00,627.00,"8,330.00"
3,離脱,6,"10,660.00",1.50,712.00,"6,905.00"
4,その他,4,"9,720.00",2.30,510.00,"4,373.00"
5,新規有望,4,"4,650.00",1.50,479.00,"3,213.00"


### 05. 離脱リスク顧客一覧

**ビジネス課題:** 最終購買から90日以上の顧客を抽出し、フォロー対象を特定する。

**使用テクニック:** `HAVING`, `DATEDIFF`, `CURDATE()`

In [6]:
sql = '''
WITH customer_activity AS (
    SELECT
        o.customer_id,
        MAX(o.order_date)                       AS last_order_date,
        DATEDIFF(CURDATE(), MAX(o.order_date))  AS days_since_last,
        COUNT(DISTINCT o.order_id)              AS total_orders,
        SUM(oi.quantity * oi.unit_price)         AS total_spent
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.status IN ('paid', 'shipped')
    GROUP BY o.customer_id
    HAVING DATEDIFF(CURDATE(), MAX(o.order_date)) >= 90
)
SELECT
    c.customer_id,
    c.customer_name,
    c.email,
    ca.last_order_date,
    ca.days_since_last,
    ca.total_orders,
    ca.total_spent,
    CASE
        WHEN ca.total_spent >= 30000 THEN '高LTV — 優先フォロー'
        WHEN ca.total_orders >= 3    THEN 'リピーター — 再活性化施策'
        ELSE '一般 — 標準リマインド'
    END AS follow_up_priority
FROM customer_activity ca
JOIN customers c ON ca.customer_id = c.customer_id
ORDER BY ca.total_spent DESC;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_26396\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,customer_id,customer_name,email,last_order_date,days_since_last,total_orders,total_spent,follow_up_priority
0,29,小林誠,user29@example.com,2024-11-23 22:09:00,495,6,"59,900.00",高LTV — 優先フォロー
1,1,田中太郎,user01@example.com,2024-12-22 11:25:00,466,3,"57,500.00",高LTV — 優先フォロー
2,5,高橋翔太,user05@example.com,2024-11-02 12:23:00,516,5,"50,100.00",高LTV — 優先フォロー
3,13,松本健二,user13@example.com,2024-05-22 18:23:00,680,4,"43,400.00",高LTV — 優先フォロー
4,4,山田大輔,user04@example.com,2024-06-08 11:04:00,663,3,"42,500.00",高LTV — 優先フォロー
5,17,清水拓也,user17@example.com,2024-10-09 10:59:00,540,3,"23,800.00",リピーター — 再活性化施策
6,27,伊藤遥,user27@example.com,2024-11-12 17:35:00,506,4,"22,100.00",リピーター — 再活性化施策
7,25,高橋翔太,user25@example.com,2024-04-17 22:00:00,715,2,"22,000.00",一般 — 標準リマインド
8,8,中村和也,user08@example.com,2024-06-09 18:31:00,662,3,"21,960.00",リピーター — 再活性化施策
9,20,前田浩,user20@example.com,2024-10-19 09:04:00,530,4,"21,560.00",リピーター — 再活性化施策


---

In [7]:
con.close()
print('接続を閉じました。')

接続を閉じました。
